# 04 — Dataset Distillation

Uses Cosmos-Reason2-8B as a teacher to regenerate high-quality open-ended answers for training pairs, replacing the original programmatic labels which contained annotation noise.

**What this does:** For `_scene`, `_hazard`, `_obj_id`, and `_spatial` QA types, the 8B model watches each video and generates a proper answer. MCQ answers (binary attributes + categorical) are kept as-is since they come directly from verified dataset annotations.

**Output:** `underwater_vqa_train_distilled.json` and `underwater_vqa_val_distilled.json` — used by notebook 05 for SFT training.

**Run order:** Cell 1 (install) → Cell 2 (mount + login + download videos + load 8B) → Cell 3 (helpers) → Cell 4 (distill train) → Cell 5 (distill val) → Cell 6 (preview + clean) → Cell 7 (apply cleaning + save)

## Cell 1 — Install

In [ ]:
!pip install decord fiftyone -q

## Cell 2 — Mount Drive, Login & Load 8B Model

Loads Cosmos-Reason2-8B as the teacher model. Requires ~20GB VRAM — H100 recommended.

Also downloads WebUOT-238-Test videos via fiftyone if not already present in this session.

In [ ]:
from google.colab import drive, userdata
from huggingface_hub import login

drive.mount('/content/drive', force_remount=True)
login(token=userdata.get("HF_TOKEN"))

import re, torch, json, os, numpy as np
from PIL import Image
import decord
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
import fiftyone as fo
from fiftyone.utils.huggingface import load_from_hub

# ── Video path ────────────────────────────────────────────────────────────────
VIDEO_DIR = "/root/fiftyone/huggingface/hub/Voxel51/WebUOT-238-Test/data"

def fix_video_path(p):
    """Ensure video path points to local fiftyone copy."""
    return os.path.join(VIDEO_DIR, os.path.basename(p))

# Download videos if not present in this session
if not os.path.exists(VIDEO_DIR) or not os.listdir(VIDEO_DIR):
    print("Videos not found locally — downloading from HuggingFace...")
    load_from_hub("Voxel51/WebUOT-238-Test", overwrite=False)
    print("Download complete.")
else:
    print(f"Videos already present: {len(os.listdir(VIDEO_DIR))} files")

# ── Load 8B teacher model ─────────────────────────────────────────────────────
MODEL_8B = "nvidia/Cosmos-Reason2-8B"

processor_8b = AutoProcessor.from_pretrained(MODEL_8B, trust_remote_code=True)
processor_8b.image_processor.min_pixels = 256 * 28 * 28
processor_8b.image_processor.max_pixels = 512 * 28 * 28

print("Loading Cosmos-Reason2-8B teacher model...")
model_8b = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_8B,
    dtype=torch.bfloat16,
    device_map="auto",
)
model_8b.eval()
print(f"8B model loaded. VRAM: {torch.cuda.memory_reserved()/1024**3:.1f} GB")

## Cell 3 — Helpers

`extract_frames` samples 8 evenly-spaced frames from a video. `generate_ground_truth_with_8b` queries the teacher model with an AUV-specific system prompt to produce a high-quality answer.

In [ ]:
N_FRAMES = 8

AUV_SYSTEM_PROMPT = (
    "You are an expert underwater robotics vision system. "
    "Analyze underwater video footage and provide precise, structured assessments "
    "for autonomous underwater vehicle (AUV) navigation. "
    "Focus on: visibility conditions, environment type, navigation hazards, "
    "and target characteristics. Be concise and factual."
)


def extract_frames(video_path, n_frames=N_FRAMES):
    vr      = decord.VideoReader(video_path, ctx=decord.cpu(0))
    indices = np.linspace(0, len(vr) - 1, n_frames, dtype=int)
    frames  = vr.get_batch(indices).asnumpy()
    return [Image.fromarray(frames[i]) for i in range(n_frames)]


def generate_ground_truth_with_8b(model, processor, video_path, question):
    """Use Cosmos-Reason2-8B to generate a high-quality ground truth answer."""
    try:
        frames = extract_frames(video_path)
    except Exception:
        return None

    messages = [
        {"role": "system", "content": AUV_SYSTEM_PROMPT},
        {"role": "user", "content": [
            *[{"type": "image", "image": f} for f in frames],
            {"type": "text",  "text": question},
        ]},
    ]
    text   = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=frames, return_tensors="pt", padding=True)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        out_ids = model.generate(**inputs, max_new_tokens=256, do_sample=False)

    generated = out_ids[0][inputs["input_ids"].shape[1]:]
    return processor.decode(generated, skip_special_tokens=True).strip()


print("Helpers defined.")

## Cell 4 — Distill training set

Regenerates open-ended answers (`_scene`, `_hazard`, `_obj_id`, `_spatial`) using the 8B teacher. MCQ samples are kept as-is. Saves intermediate output to Drive.

Expected: ~2–3 hours on H100 for 2,023 samples.

In [ ]:
from tqdm import tqdm

TRAIN_PATH     = "/content/drive/MyDrive/cosmos-cookoff/data/underwater_vqa_train.json"
DISTILLED_PATH = "/content/drive/MyDrive/cosmos-cookoff/data/underwater_vqa_train_distilled.json"

with open(TRAIN_PATH) as f:
    train_data = json.load(f)

# Patch all video paths to local fiftyone copies
for item in train_data:
    item["video"] = fix_video_path(item["video"])

print(f"Loaded {len(train_data)} training pairs")
print(f"Sample video exists: {os.path.exists(train_data[0]['video'])}")
print(f"Sample path: {train_data[0]['video']}")

# Open-ended types have label noise — regenerate with 8B
# MCQ labels come from verified dataset annotations — keep as-is
REGEN_TYPES = ["_scene", "_hazard", "_obj_id", "_spatial"]

distilled_train = []
regenerated = 0
kept        = 0
failed      = 0

for item in tqdm(train_data, desc="Distilling train with 8B"):
    needs_regen = any(t in item["id"] for t in REGEN_TYPES)

    if not needs_regen:
        distilled_train.append(item)
        kept += 1
        continue

    question   = item["conversations"][0]["value"].replace("<video>\n", "").strip()
    new_answer = generate_ground_truth_with_8b(model_8b, processor_8b, item["video"], question)

    if new_answer is None:
        failed += 1  # Unreadable video — drop sample
        continue

    distilled_train.append({
        "id":    item["id"],
        "video": item["video"],
        "conversations": [
            item["conversations"][0],             # Keep original question
            {"from": "gpt", "value": new_answer}  # Replace with 8B answer
        ]
    })
    regenerated += 1

    if regenerated % 50 == 0:
        print(f"  Regenerated: {regenerated} | Kept: {kept} | Failed: {failed}")

print(f"\nDone.")
print(f"Regenerated with 8B: {regenerated}")
print(f"Kept original MCQ:   {kept}")
print(f"Failed/dropped:      {failed}")
print(f"Total:               {len(distilled_train)}")

with open(DISTILLED_PATH, "w") as f:
    json.dump(distilled_train, f, indent=2)
print(f"Saved intermediate to {DISTILLED_PATH}")

## Cell 5 — Distill validation set

Same process for the val set. Used as ground truth in notebook 06 evaluation (ROUGE-L scoring).

In [ ]:
VAL_PATH           = "/content/drive/MyDrive/cosmos-cookoff/data/underwater_vqa_val.json"
DISTILLED_VAL_PATH = "/content/drive/MyDrive/cosmos-cookoff/data/underwater_vqa_val_distilled.json"

with open(VAL_PATH) as f:
    val_data = json.load(f)

# Patch all video paths to local fiftyone copies
for item in val_data:
    item["video"] = fix_video_path(item["video"])

print(f"Loaded {len(val_data)} val pairs")
print(f"Sample video exists: {os.path.exists(val_data[0]['video'])}")

distilled_val = []
regenerated   = 0
kept          = 0

for item in tqdm(val_data, desc="Distilling val with 8B"):
    needs_regen = any(t in item["id"] for t in REGEN_TYPES)

    if not needs_regen:
        distilled_val.append(item)
        kept += 1
        continue

    question   = item["conversations"][0]["value"].replace("<video>\n", "").strip()
    new_answer = generate_ground_truth_with_8b(model_8b, processor_8b, item["video"], question)

    if new_answer is None:
        distilled_val.append(item)  # Keep original on failure
        kept += 1
        continue

    # Strip markdown from val answers (consistent clean format for ROUGE-L scoring)
    cleaned = re.sub(r'#{1,4}\s*', '', new_answer)
    cleaned = re.sub(r'\*\*(.*?)\*\*', r'\1', cleaned)
    cleaned = re.sub(r'\*(.*?)\*', r'\1', cleaned)
    cleaned = re.sub(r'^\s*[-•]\s*', '', cleaned, flags=re.MULTILINE)
    cleaned = re.sub(r'\n+', ' ', cleaned)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()

    distilled_val.append({
        "id":    item["id"],
        "video": item["video"],
        "conversations": [
            item["conversations"][0],
            {"from": "gpt", "value": cleaned}
        ]
    })
    regenerated += 1

print(f"\nRegenerated: {regenerated} | Kept: {kept} | Total: {len(distilled_val)}")

with open(DISTILLED_VAL_PATH, "w") as f:
    json.dump(distilled_val, f, indent=2)
print(f"Saved to {DISTILLED_VAL_PATH}")

## Cell 6 — Preview & define answer cleaner

Inspect raw 8B outputs for `_scene` samples, then define `clean_8b_answer` which normalises them into a consistent structured format. Scene answers especially need this because the 8B tends to produce markdown headers and bullet points.

In [ ]:
# Preview 5 raw 8B scene answers before cleaning
regen_samples = [x for x in distilled_train if "_scene" in x["id"]][:5]
for s in regen_samples:
    print(f"\nVideo: {s['id']}")
    print(f"Q: {s['conversations'][0]['value'][:80]}")
    print(f"A: {s['conversations'][1]['value'][:150]}")


def clean_8b_answer(raw_answer):
    """
    Normalise a raw 8B scene answer into a clean structured format.
    Strips markdown, infers visibility level and environment, and
    returns a single sentence consistent with the training format.
    """
    text = re.sub(r'#{1,4}\s*', '', raw_answer)
    text = re.sub(r'\*\*(.*?)\*\*', r'\1', text)
    text = re.sub(r'\*(.*?)\*', r'\1', text)
    text = re.sub(r'^\s*[-•]\s*', '', text, flags=re.MULTILINE)
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text_lower = text.lower()

    if re.search(r'\b(low|poor|murky|turbid|limited visibility)\b', text_lower):
        nav_assessment = "Underwater visibility is low, which presents significant challenges for AUV navigation — the camera range is limited and target detection is difficult."
    elif re.search(r'\b(moderate|medium|hazy|reduced)\b', text_lower):
        nav_assessment = "Underwater visibility is moderate, allowing for adequate AUV navigation with some limitations in detection range."
    elif re.search(r'\b(clear|good|high|excellent)\b', text_lower):
        nav_assessment = "Underwater visibility is good, providing suitable conditions for AUV navigation and target detection."
    else:
        nav_assessment = "Underwater visibility conditions present challenges for AUV navigation."

    env = "sea"
    for keyword, label in [
        ("river", "river"), ("lake", "lake"),
        ("fish tank", "fish tank"), ("aquarium", "fish tank"),
        ("ocean", "sea"), ("reef", "sea"), ("coral", "sea"),
    ]:
        if keyword in text_lower:
            env = label
            break

    return f"This footage is from a {env} environment. {nav_assessment}"


# Test cleaner on the 5 preview samples
print("\n--- Cleaning test ---")
for s in regen_samples:
    raw     = s["conversations"][1]["value"]
    cleaned = clean_8b_answer(raw)
    print(f"\nRaw:     {raw[:80]}")
    print(f"Cleaned: {cleaned}")

## Cell 7 — Apply cleaning & save final distilled train

Applies `clean_8b_answer` to all `_scene` samples and strips markdown from all other regenerated types. Overwrites the intermediate file with the final clean version.

In [ ]:
FINAL_DISTILLED_PATH = "/content/drive/MyDrive/cosmos-cookoff/data/underwater_vqa_train_distilled.json"

final_train   = []
cleaned_count = 0

for item in distilled_train:
    needs_clean = any(t in item["id"] for t in REGEN_TYPES)

    if not needs_clean:
        final_train.append(item)
        continue

    raw = item["conversations"][1]["value"]

    if "_scene" in item["id"]:
        cleaned = clean_8b_answer(raw)
    else:
        cleaned = re.sub(r'#{1,4}\s*', '', raw)
        cleaned = re.sub(r'\*\*(.*?)\*\*', r'\1', cleaned)
        cleaned = re.sub(r'\*(.*?)\*', r'\1', cleaned)
        cleaned = re.sub(r'^\s*[-•]\s*', '', cleaned, flags=re.MULTILINE)
        cleaned = re.sub(r'\n+', ' ', cleaned)
        cleaned = re.sub(r'\s+', ' ', cleaned).strip()

    final_train.append({
        "id":    item["id"],
        "video": item["video"],
        "conversations": [
            item["conversations"][0],
            {"from": "gpt", "value": cleaned}
        ]
    })
    cleaned_count += 1

print(f"Cleaned {cleaned_count} samples")

# Spot-check one of each type
for t in ["_scene", "_hazard", "_obj_id", "_spatial"]:
    sample = next((x for x in final_train if t in x["id"]), None)
    if sample:
        print(f"\n{t}: {sample['conversations'][1]['value'][:150]}")

with open(FINAL_DISTILLED_PATH, "w") as f:
    json.dump(final_train, f, indent=2)
print(f"\nSaved {len(final_train)} samples to {FINAL_DISTILLED_PATH}")